<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/quant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Quantization from scratch

In [229]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import urllib.request as req
from transformers import AutoTokenizer, AutoModelForCausalLM

In [230]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [231]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [232]:
class INTQuantizer():
  @staticmethod
  def quantize_int8_per_tensor(tensor: torch.Tensor):
    scale = tensor.abs().max() / 127
    scale = scale.clamp(min=1e-8)
    out = torch.round(tensor / scale).clamp(-128, 127).to(torch.int8)
    return out, scale

  @staticmethod
  def dequantize_int8_per_tensor(q_tensor: torch.Tensor, scale: torch.Tensor):
    return q_tensor.float() * scale

  @staticmethod
  def quantize_int8_per_channel(tensor: torch.Tensor): # quantizes/scales per row
    scales = tensor.abs().amax(dim=1, keepdim=True) / 127
    scales = scales.clamp(min=1e-8)
    out = torch.round(tensor / scales).clamp(-127, 127).to(torch.int8)
    return out, scales

  @staticmethod
  def dequantize_int8_per_channel(q_tensor: torch.Tensor, scales: torch.Tensor):
    return q_tensor.float() * scales

  @staticmethod
  def quantize_int4_groupwise(tensor: torch.Tensor, group_size = 128):
    out_dim, in_dim = tensor.shape
    pad = (group_size - (in_dim % group_size)) % group_size
    if pad:
      T_padded = F.pad(tensor, (0, pad))
    else:
      T_padded = tensor
    padded_dim = T_padded.shape[1]
    T_reshaped = T_padded.view(out_dim, padded_dim // group_size, group_size)

    scales = T_reshaped.abs().amax(dim=2, keepdim=True) / 7
    scales = scales.clamp(min=1e-8)
    out = torch.round(T_reshaped / scales).clamp(-8, 7).to(torch.int8)

    meta = {
        "original_shape": tensor.shape,
        "padded_shape": T_padded.shape,
        "group_size": group_size,
        "pad": pad
    }

    return out, scales, meta

  @staticmethod
  def dequantize_int4_groupwise(q_tensor: torch.Tensor, scales: torch.Tensor, meta: dict):
    T_padded_hat = (q_tensor.float() * scales).view(meta["padded_shape"])
    out_dim, in_dim = meta["original_shape"]
    return T_padded_hat[:, :in_dim]

  @staticmethod
  def tensor_error(tensor: torch.Tensor, tensor_hat: torch.Tensor):
    mse = torch.mean((tensor - tensor_hat) ** 2).item()
    mae = torch.mean(torch.abs(tensor - tensor_hat)).item()
    max_err = torch.max(torch.abs(tensor -tensor_hat)).item()
    rel_mse = mse / (torch.mean(tensor ** 2).item() + 1e-12)
    return mse, mae, max_err, rel_mse

Test INTQuantizer class

In [233]:
iq = INTQuantizer()
test_tensor = torch.tensor([-10.0, 0.0, 5.0, 10.0])

quantized_tensor, scale = iq.quantize_int8_per_tensor(test_tensor)
assert torch.isclose(scale, torch.tensor(10.0/127.0))

dequantized_tensor = iq.dequantize_int8_per_tensor(quantized_tensor, scale)
assert torch.allclose(test_tensor, dequantized_tensor, atol=scale.item()/2)

In [234]:
test_tensor = torch.tensor([[-10., 0., 5., 10.], [-20., -5., 0., 15.]])
quantized_tensor, scales = iq.quantize_int8_per_channel(test_tensor)
dequantized_tensor = iq.dequantize_int8_per_channel(quantized_tensor, scales)
max_atol = scales.max().item() / 2
assert torch.allclose(test_tensor, dequantized_tensor, atol=max_atol)

In [235]:
test_tensor_groupwise = torch.randn(10, 256) * 10
group_size = 64

quantized_tensor_groupwise, scales_groupwise, meta_groupwise = iq.quantize_int4_groupwise(test_tensor_groupwise, group_size=group_size)
dequantized_tensor_groupwise = iq.dequantize_int4_groupwise(quantized_tensor_groupwise, scales_groupwise, meta_groupwise)
max_atol_groupwise = scales_groupwise.max().item() / 2
assert torch.allclose(test_tensor_groupwise, dequantized_tensor_groupwise, atol=max_atol_groupwise)

Quantizing Linear Layers

In [236]:
class QuantLinearInt8PerTensor(nn.Module):
  def __init__(self, linear: nn.Linear):
    super().__init__()
    qweight, scale = iq.quantize_int8_per_tensor(linear.weight)
    self.register_buffer("qweight", qweight)
    self.register_buffer("scale", scale)
    if linear.bias is not None:
      self.register_buffer("bias", linear.bias.data.clone())
    else:
      self.bias = None

  def forward(self, x: torch.Tensor):
    W = iq.dequantize_int8_per_tensor(self.qweight, self.scale).to(x.dtype)
    return F.linear(x, W, self.bias)

In [237]:
class QuantLinearInt8PerChannel(nn.Module):
  def __init__(self, linear: nn.Linear):
    super().__init__()
    qweight, scales = iq.quantize_int8_per_channel(linear.weight)
    self.register_buffer("qweight", qweight)
    self.register_buffer("scales", scales)
    if linear.bias is not None:
      self.register_buffer("bias", linear.bias.data.clone())
    else:
      self.bias = None

  def forward(self, x: torch.Tensor):
    W = iq.dequantize_int8_per_channel(self.qweight, self.scales).to(x.dtype)
    return F.linear(x, W, self.bias)

In [238]:
class QuantLinearInt4Groupwise(nn.Module):
  def __init__(self, linear: nn.Linear, group_size: int):
    super().__init__()
    qweight, scales, meta = iq.quantize_int4_groupwise(linear.weight, group_size)
    self.register_buffer("qweight", qweight)
    self.register_buffer("scales", scales)
    self.meta = meta
    if linear.bias is not None:
      self.register_buffer("bias", linear.bias.data.clone())
    else:
      self.bias = None

  def forward(self, x: torch.Tensor):
    W = iq.dequantize_int4_groupwise(self.qweight, self.scales, self.meta).to(x.dtype)
    return F.linear(x, W, self.bias)

Testing Quantizing Class

In [239]:
linear_layer = nn.Linear(10, 5)
quant_linear_layer = QuantLinearInt8PerTensor(linear_layer)

assert quant_linear_layer.qweight.dtype == torch.int8, "qweight should be int8"
assert quant_linear_layer.scale.dtype == torch.float, "scale should be float"

In [240]:
linear_layer_channel = nn.Linear(10, 5)
quant_linear_layer_channel = QuantLinearInt8PerChannel(linear_layer_channel)

assert quant_linear_layer_channel.qweight.dtype == torch.int8, "qweight should be int8 for per-channel"
assert quant_linear_layer_channel.scales.dtype == torch.float, "scales should be float for per-channel"

In [241]:
linear_layer_groupwise = nn.Linear(10, 5)
quant_linear_layer_groupwise = QuantLinearInt4Groupwise(linear_layer_groupwise, group_size=5)

assert quant_linear_layer_groupwise.qweight.dtype == torch.int8, "qweight should be int8 for groupwise"
assert quant_linear_layer_groupwise.scales.dtype == torch.float, "scales should be float for groupwise"
assert isinstance(quant_linear_layer_groupwise.meta, dict), "meta should be a dictionary for groupwise"

Replace Linear Layers with Quantized layers

In [242]:
def replace_linear_layers(module: nn.Module, scheme, group_size=128):
  for name, child in list(module.named_children()):
    if isinstance(child, nn.Linear):
      if scheme == "int8_per_tensor":
        setattr(module, name, QuantLinearInt8PerTensor(child))
      elif scheme == "int8_per_channel":
        setattr(module, name, QuantLinearInt8PerChannel(child))
      elif scheme == "int4_groupwise":
        setattr(module, name, QuantLinearInt4Groupwise(child, group_size))
      else:
        raise ValueError(f"Invalid scheme: {scheme}")
    else:
      replace_linear_layers(child, scheme, group_size)
  return module

In [243]:
import copy

def make_quantized_model(base_model, scheme, group_size=128):
  model = copy.deepcopy(base_model)
  model = replace_linear_layers(model, scheme, group_size)
  return model

Loading Text

In [244]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

try:
    with req.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    print("Total Length", len(raw_text))
except:
    print("Text download failed")

Total Length 1115394


Calculating Loss && Perplexity

In [245]:
@torch.no_grad()
def perplexity(model: nn.Module, texts, max_length=256):
  model.eval()

  enc = tokenizer(
      texts,
      return_tensors="pt",
      padding=True,
      truncation=True,
      max_length=max_length
  ).to(device)

  input_ids = enc["input_ids"]
  attention_mask = enc["attention_mask"]
  labels = input_ids.clone()
  labels[attention_mask == 0] = -100

  out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
  loss = out.loss
  perplexity = torch.exp(loss)
  return loss, perplexity

In [246]:
baseline_loss, baseline_perplexity = perplexity(model, raw_text)
print(f"Baseline Loss: {baseline_loss.item()}, Baseline Perplexity: {baseline_perplexity.item()}")

Baseline Loss: 3.333599090576172, Baseline Perplexity: 28.03907585144043


Test the Perplexity of Quantized models

In [247]:
q_model_int8_per_tensor = make_quantized_model(model, "int8_per_tensor")
q_model_int8_per_tensor = q_model_int8_per_tensor.to(device)
q_loss_int8_per_tensor, q_perplexity_int8_per_tensor = perplexity(q_model_int8_per_tensor, raw_text)
print(f"Quantized Loss (int8 per tensor): {q_loss_int8_per_tensor.item()}, Quantized Perplexity (int8 per tensor): {q_perplexity_int8_per_tensor.item()}")

Quantized Loss (int8 per tensor): 3.407289743423462, Quantized Perplexity (int8 per tensor): 30.18332862854004


In [248]:
q_model_int8_per_channel = make_quantized_model(model, "int8_per_channel")
q_model_int8_per_channel = q_model_int8_per_channel.to(device)
q_loss_int8_per_channel, q_perplexity_int8_per_channel = perplexity(q_model_int8_per_channel, raw_text)
print(f"Quantized Loss (int8 per channel): {q_loss_int8_per_channel.item()}, Quantized Perplexity (int8 per channel): {q_perplexity_int8_per_channel.item()}")

Quantized Loss (int8 per channel): 3.329045534133911, Quantized Perplexity (int8 per channel): 27.91168785095215


In [249]:
q_model_int4_groupwise = make_quantized_model(model, "int4_groupwise")
q_model_int4_groupwise = q_model_int4_groupwise.to(device)
q_loss_int4_groupwise, q_perplexity_int4_groupwise = perplexity(q_model_int4_groupwise, raw_text)
print(f"Quantized Loss (int4 groupwise): {q_loss_int4_groupwise.item()}, Quantized Perplexity (int4 groupwise): {q_perplexity_int4_groupwise.item()}")

Quantized Loss (int4 groupwise): 3.9061226844787598, Quantized Perplexity (int4 groupwise): 49.70585250854492
